In [ ]:
using Pkg
toktox_path = "/Users/adrianaghiozzi/Code/Sources/dev/TokToxStability"   # adjust to match the location of your cloned TokToxStability repo
Pkg.activate(toktox_path)
Pkg.instantiate()

In [ ]:
venv_python = joinpath(toktox_path, ".venv", "bin", "python")
if !isfile(venv_python)
    run(`$(joinpath(toktox_path, "setup_venv.sh"))`)
end

In [ ]:
ENV["JULIA_CONDAPKG_BACKEND"] = "Null"
ENV["JULIA_PYTHONCALL_EXE"] = venv_python

In [ ]:
using Revise
using TokToxStability
using Plots
using DataFrames
using PythonCall

In [ ]:
# only call this once per kernel session
myOFT = TokToxStability.make_oft_env()   


In [ ]:
# An example kinetic EFIT and corresponding geqdsk for DIII-D shot 147131 are provided in cases/D3D
gpec_settings = TokToxStability.GPECSettings(toroidal_mode_number = 1)
kinetic_efit_path = joinpath(pkgdir(TokToxStability), "cases/D3D/g147131.02300_DIIID_KEFIT.kin")
geqdsk_path = joinpath(pkgdir(TokToxStability), "cases/D3D/g147131.02300_DIIID_KEFIT.eqdsk")


In [ ]:
profiles = TokToxStability.read_kinetic_profiles_from_file(kinetic_efit_path)
equilibrium = TokToxStability.read_equilibrium_from_file(geqdsk_path)


In [ ]:
mygs = TokToxStability.setup_tokamaker_from_file!(:D3D, profiles, equilibrium, myOFT)


In [ ]:
# The gEQDSK files created by TokaMaker at each coarse scan iteration can be saved by setting working_directory
# For a file-less option, set working_directory = nothing, which will save the gEQDSKs to a temporary directory that is deleted after the run
working_directory = nothing

# starting_fraction sets the first scaling of the temperature profile to be applied in the beta limit search
# step sets the spread of the subsequent scalings 
starting_fraction = 1.0 
step = 0.25 

coarse_scan_results = TokToxStability.coarse_scan_until_unstable(mygs, starting_fraction, step, equilibrium, profiles, gpec_settings; working_directory = working_directory);


In [ ]:
# Creates a DataFrame with the results of the scan 
cols = (:fraction, :beta_n, :dW_total, :dW_plasma, :dW_vacuum, :qmax, :msing)
DataFrame([NamedTuple{cols}(r[c] for c in cols) for r in coarse_scan_results])

In [ ]:
# estimate_beta_limit function interpolates the zero crossing of dW_total based on the scanned points
# If the limit wasn't reached, try re-running coarse_scan_until_unstable with a larger starting_fraction
bl = TokToxStability.estimate_beta_limit(coarse_scan_results)


In [ ]:
TokToxStability.plot_beta_scan_summary(coarse_scan_results; beta_limit=bl)


In [ ]:
TokToxStability.plot_profiles(coarse_scan_results)
